In [6]:
# 导入必需的包
from langchain_community.document_loaders import UnstructuredExcelLoader, Docx2txtLoader, PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os  # 补充：用于获取环境变量
import dotenv
dotenv.load_dotenv()


# 定义 chatdoc
class ChatDoc:
    def __init__(self):
        self.doc = None
        self.splitText = []  # 分割后的文本 split text
        self.template = [
            ("system", "你是一个处理文档的秘书,你从不说自己是一个大模型或者AI助手,你会根据下面提供的上下文内容来继续回答问题.\n 上下文内容\n {context} \n"),
            ("human", "你好！ "),
            ("ai", "您好，我是尚硅谷秘书"),
            ("human", "{question}"),
        ]
        self.prompt = ChatPromptTemplate.from_messages(self.template)

    def getFile(self):
        doc = self.doc
        loaders = {
            "docx": Docx2txtLoader,
            "pdf": PyPDFLoader,
            "xlsx": UnstructuredExcelLoader,
        }
        file_extension = doc.split(".")[-1]
        loader_class = loaders.get(file_extension)
        if loader_class:
            try:
                loader = loader_class(doc)
                text = loader.load()
                return text
            except Exception as e:
                print(f"Error loading {file_extension} files: {e}")
        else:
            print(f"Unsupported file extension: {file_extension}")
            return None

    def splitSentences(self):
        full_text = self.getFile()  # 获取文档内容 get the content of the document
        if full_text is not None:
            # 对文档进行分割
            text_split = CharacterTextSplitter(
                chunk_size=500,
                chunk_overlap=100,
                separator="\n\n",
                length_function=len,
                is_separator_regex=False
            )
            texts = text_split.split_documents(full_text)
            self.splitText = texts

    def embeddingAndVectorDB(self):
        # 使用 BAAI/bge-m3 模型（已注释，若需使用请取消注释）
        # embeddings = OpenAIEmbeddings(
        #     model="BAAI/bge-m3",
        #     api_key=os.getenv("SILICON_API_KEY"),
        #     base_url="https://api.siliconflow.cn/v1"
        # )

        os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
        os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
        embeddings = OpenAIEmbeddings()

        db = Chroma.from_documents(
            documents=self.splitText,
            embedding=embeddings,
        )
        return db

    def askAndFindFiles(self, question):
        db = self.embeddingAndVectorDB()
        #print(db.collection.count())
        retriever = db.as_retriever()
        return retriever.invoke(input=question)

    def chatWithDoc(self, question):
        _content = ""
        context = self.askAndFindFiles(question)
        for i in context:
            _content += i.page_content
        print(f"_content: {_content}")
        messages = self.prompt.format_messages(context=_content, question=question)

        print("messages:", messages)

        llm = ChatOpenAI(
            model="gpt-4",
            temperature=0,
            api_key=os.getenv("OPENAI_API_KEY1"),
            base_url=os.getenv("OPENAI_BASE_URL")
        )
        return llm.invoke(messages)


# 示例使用
chat_doc = ChatDoc()
chat_doc.doc = "./asset/load/13-sgg_chat.docx"
chat_doc.splitSentences()
response = chat_doc.chatWithDoc("尚硅地址在哪")
print(response.content)

尚硅谷教育科技有限公司的总部地址位于北京市海淀区中关村软件园创新大厦B座5层。
